In [1]:
!pip install -q pyspark


In [2]:
import os, shutil, glob
for p in ["data_lake/bronze", "data_lake/silver/preprocessed_sales"]:
    os.makedirs(p, exist_ok=True)
try:
    from google.colab import files
    src = next(iter(files.upload()))
except ImportError:
    src = "sales.csv"
shutil.copy(src, "data_lake/bronze/sales.csv")
for root, _, fs in os.walk("data_lake"):
    print(root, fs)

Saving sales.csv to sales.csv
data_lake []
data_lake/silver []
data_lake/silver/preprocessed_sales []
data_lake/bronze ['sales.csv']


In [3]:
from pyspark.sql import SparkSession, functions as F
spark = SparkSession.builder.appName("SalesDataLake").master("local[*]").config("spark.sql.ansi.enabled", "false").getOrCreate()
raw = spark.read.option("header", True).option("inferSchema", True).csv("data_lake/bronze/sales.csv")
n_before = raw.count()
raw.printSchema()
raw.show(5, truncate=False)
print("Total records:", n_before)

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- discount_percent: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_status: string (nullable = true)

+--------+-----------+--------------+-------------+--------+--------+----------+----------------+--------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name |product      |category|quantity|unit_price|discount_percent|payment_method|city      |state         |order_date|order_status|
+--------+-----------+--------------+-------------+--------+--------+----------+----------------+--------------+----------

In [4]:
str_cols = [c for c, t in raw.dtypes if t == "string"]
text_cols = ["customer_name", "product", "category", "payment_method", "city", "state", "order_status"]
print("Exact duplicate rows:", n_before - raw.dropDuplicates().count())
print("Rows sharing an order_id with another row:", n_before - raw.select("order_id").distinct().count())
print("Null or blank values per column")
raw.select([F.sum((F.col(c).isNull() | (F.trim(F.col(c).cast("string")) == "")).cast("int")).alias(c) for c in raw.columns]).show(vertical=True)

Exact duplicate rows: 10
Rows sharing an order_id with another row: 18
Null or blank values per column
-RECORD 0---------------
 order_id         | 0   
 customer_id      | 0   
 customer_name    | 10  
 product          | 0   
 category         | 0   
 quantity         | 0   
 unit_price       | 0   
 discount_percent | 0   
 payment_method   | 8   
 city             | 8   
 state            | 0   
 order_date       | 0   
 order_status     | 0   



In [5]:
print("Text fields: rows with extra or leading/trailing spaces")
raw.select([F.sum((F.col(c) != F.trim(F.regexp_replace(F.col(c), r"\s+", " "))).cast("int")).alias(c) for c in text_cols]).show(vertical=True)
print("Text fields: distinct values raw vs after trim and lowercase")
raw.select([F.countDistinct(c).alias(c + "_raw") for c in text_cols] + [F.countDistinct(F.lower(F.trim(c))).alias(c + "_norm") for c in text_cols]).show(vertical=True)
raw.select("category").distinct().orderBy("category").show(20)

Text fields: rows with extra or leading/trailing spaces
-RECORD 0-------------
 customer_name  | 3   
 product        | 0   
 category       | 0   
 payment_method | 0   
 city           | 3   
 state          | 0   
 order_status   | 0   

Text fields: distinct values raw vs after trim and lowercase
-RECORD 0------------------
 customer_name_raw   | 305 
 product_raw         | 57  
 category_raw        | 13  
 payment_method_raw  | 5   
 city_raw            | 18  
 state_raw           | 9   
 order_status_raw    | 5   
 customer_name_norm  | 302 
 product_norm        | 57  
 category_norm       | 8   
 payment_method_norm | 5   
 city_norm           | 11  
 state_norm          | 9   
 order_status_norm   | 5   

+---------------+
|       category|
+---------------+
|         Beauty|
|          Books|
|       CLOTHING|
|       Clothing|
|    ELECTRONICS|
|    Electronics|
|      Furniture|
|        Grocery|
|Home Appliances|
|         SPORTS|
|         Sports|
|         beauty|
|      

In [6]:
print("Invalid numeric values")
raw.select(
    F.sum((F.col("quantity") <= 0).cast("int")).alias("quantity_le_0"),
    F.sum((F.col("unit_price") <= 0).cast("int")).alias("unit_price_le_0"),
    F.sum(((F.col("discount_percent") < 0) | (F.col("discount_percent") > 100)).cast("int")).alias("discount_outside_0_100")
).show()

Invalid numeric values
+-------------+---------------+----------------------+
|quantity_le_0|unit_price_le_0|discount_outside_0_100|
+-------------+---------------+----------------------+
|           12|              8|                     8|
+-------------+---------------+----------------------+



In [7]:
fmts = {"yyyy-MM-dd": r"^\d{4}-\d{2}-\d{2}$", "yyyy/MM/dd": r"^\d{4}/\d{2}/\d{2}$", "dd/MM/yyyy": r"^\d{2}/\d{2}/\d{4}$", "MM-dd-yyyy": r"^\d{2}-\d{2}-\d{4}$"}
def parse_date(c):
    s = F.trim(F.col(c).cast("string"))
    return F.coalesce(*[F.when(s.rlike(p), F.to_date(s, f)) for f, p in fmts.items()])
dates = raw.select("order_id", "order_date", parse_date("order_date").alias("parsed"))
dates.groupBy(F.regexp_replace("order_date", r"\d", "9").alias("date_pattern")).count().show()
print("Dates that cannot be parsed:")
dates.filter(F.col("parsed").isNull()).show()

+------------+-----+
|date_pattern|count|
+------------+-----+
|  not-a-date|    1|
|  99-99-9999|    1|
|  99/99/9999|    1|
|  9999-99-99|  996|
|  9999/99/99|    1|
+------------+-----+

Dates that cannot be parsed:
+--------+----------+------+
|order_id|order_date|parsed|
+--------+----------+------+
|  100097|31/02/2026|  NULL|
|  100448|not-a-date|  NULL|
|  100634|2025-02-30|  NULL|
|  100637|2026-13-05|  NULL|
+--------+----------+------+



In [8]:
df = raw.dropDuplicates()
n_dedup = df.count()

for c in str_cols:
    df = df.withColumn(c, F.trim(F.regexp_replace(F.col(c), r"\s+", " ")))
    df = df.withColumn(c, F.when(F.col(c) == "", None).otherwise(F.col(c)))
for c in ["customer_name", "category", "city", "state"]:
    df = df.withColumn(c, F.initcap(F.col(c)))

n_filled = df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in ["customer_name", "payment_method", "city"]]).first().asDict()
df = df.fillna("Unknown", subset=["customer_name", "payment_method", "city"])

n_bad_qty = df.filter(F.col("quantity") <= 0).count()
df = df.filter(F.col("quantity") > 0)

n_bad_price = df.filter(F.col("unit_price") <= 0).count()
med = df.filter(F.col("unit_price") > 0).groupBy("product").agg(F.percentile_approx("unit_price", 0.5).alias("median_price"))
df = df.join(med, "product", "left").withColumn("unit_price", F.when(F.col("unit_price") > 0, F.col("unit_price")).otherwise(F.col("median_price"))).drop("median_price")

n_bad_disc = df.filter(~F.col("discount_percent").between(0, 100)).count()
df = df.withColumn("discount_percent", F.when(F.col("discount_percent").between(0, 100), F.col("discount_percent")).otherwise(0))

df = df.withColumn("order_date", parse_date("order_date"))
n_bad_date = df.filter(F.col("order_date").isNull()).count()

silver = df.select(raw.columns).orderBy("order_id")
n_after = silver.count()

In [9]:
silver_dir = "data_lake/silver/preprocessed_sales"
silver.coalesce(1).write.mode("overwrite").option("header", True).csv(silver_dir)
shutil.copy(glob.glob(f"{silver_dir}/part-*.csv")[0], "data_lake/silver/sales_silver.csv")
print(os.listdir(silver_dir), os.listdir("data_lake/silver"))

['._SUCCESS.crc', '.part-00000-530d1a9e-bc39-4d48-ba30-d6a39cecc67d-c000.csv.crc', '_SUCCESS', 'part-00000-530d1a9e-bc39-4d48-ba30-d6a39cecc67d-c000.csv'] ['preprocessed_sales', 'sales_silver.csv']


In [10]:
out = spark.read.option("header", True).option("inferSchema", True).csv(silver_dir)
print("Rows before:", n_before)
print("Rows after :", out.count())
out.printSchema()
out.show(10, truncate=False)


Rows before: 1000
Rows after : 978
root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- discount_percent: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- order_status: string (nullable = true)

+--------+-----------+---------------+---------------------+---------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name  |product              |category |quantity|unit_price|discount_percent|payment_method  |city      |state         |order_date|order_status|
+--------+-----------+---------------+---------------------+-----

In [11]:
print(f"Exact duplicate rows removed: {n_before - n_dedup}")
print("Whitespace trimmed and category/city/name/state capitalization standardized")
print(f"Missing values filled with 'Unknown': {n_filled}")
print(f"Rows removed for quantity <= 0: {n_bad_qty}")
print(f"unit_price <= 0 replaced with the product median: {n_bad_price}")
print(f"discount_percent outside 0-100 set to 0: {n_bad_disc}")
print(f"Dates converted to yyyy-MM-dd; unparseable dates kept as null: {n_bad_date}")
print(f"Final rows: {n_after}")

Exact duplicate rows removed: 10
Whitespace trimmed and category/city/name/state capitalization standardized
Missing values filled with 'Unknown': {'customer_name': 10, 'payment_method': 8, 'city': 8}
Rows removed for quantity <= 0: 12
unit_price <= 0 replaced with the product median: 8
discount_percent outside 0-100 set to 0: 8
Dates converted to yyyy-MM-dd; unparseable dates kept as null: 4
Final rows: 978
